# Topic 5: Broadcast Variables & Accumulators

> **Phase 4 → Week 6 → Topic 5**

---

## The Problem With Sharing Data

In distributed computing, you often need to:
1. **Share read-only reference data** with every executor (country codes, model weights, config)
2. **Aggregate counters** across all executors (rows processed, errors, skipped records)

The naive approaches are terrible:

```
# BAD: closure capture — sends a copy with EVERY task
big_lookup = {"A": 1, "B": 2, ...}  # 100MB dict
df.rdd.map(lambda row: big_lookup[row.key])  # 100MB sent for EVERY task!
# 200 tasks × 100MB = 20GB of network traffic just for the lookup!

# GOOD: broadcast variable — sent ONCE per executor, cached
bc = sc.broadcast(big_lookup)
df.rdd.map(lambda row: bc.value[row.key])  # 100MB sent once per executor
# 10 executors × 100MB = 1GB total. 20x less network traffic!
```

---

## Interview Cheat Sheet

**Q: What is a broadcast variable in Spark?**
> A broadcast variable sends a read-only copy of data to each executor node ONCE (via BitTorrent-like protocol), where it's cached in memory for all tasks on that executor to share. Without broadcast, a closure-captured variable is sent with every single task — massively wasteful for large lookup tables. Use `sc.broadcast(data)` or `F.broadcast(df)` in DataFrame joins.

**Q: What is an accumulator?**
> An accumulator is a distributed counter/aggregator that tasks can only add to (write), while the driver reads the accumulated value. Classic use: counting bad records in an ETL, tracking processed row counts, or collecting error messages. Tasks cannot read accumulators (only the driver can), which prevents coordination bottlenecks.

**Q: Can accumulators be used reliably for counting?**
> Partially. Accumulators are updated once per task in actions. In transformations (map, filter), they can be updated multiple times if tasks are retried — Spark doesn't guarantee exactly-once semantics for transformations. Always trigger accumulators from actions (not transformations) for reliable counts. Also, task retries on speculation or failure add to the count again.

**Q: What's the difference between `sc.broadcast()` and `F.broadcast()` (hint in joins)?**
> `sc.broadcast(data)` broadcasts a Python object (dict, list, etc.) to all executors — accessed via `.value` in RDD operations or UDFs. `F.broadcast(df)` is a hint to Catalyst to use BroadcastHashJoin for a DataFrame join — it doesn't create a Spark broadcast variable, it tells the optimizer to broadcast the DataFrame's data to all executors during the join physical planning.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("Week6 - Broadcast & Accumulators") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
spark.sparkContext.setLogLevel("WARN")
print("Ready")

## Part 1: Broadcast Variables for Lookup Tables

In [ ]:
# Create a lookup dict: country code → continent
country_continent = {
    "India": "Asia",    "China": "Asia",       "Japan": "Asia",
    "South Korea": "Asia", "Singapore": "Asia",
    "Germany": "Europe", "UK": "Europe",        "France": "Europe",
    "Italy": "Europe",   "Spain": "Europe",
    "USA": "America",    "Canada": "America",   "Brazil": "America",
    "Nigeria": "Africa", "Kenya": "Africa",
    "Australia": "Oceania",
}

# Broadcast the dict to all executors
bc_lookup = sc.broadcast(country_continent)

print(f"Broadcast variable created: {type(bc_lookup)}")
print(f"Value accessible via bc_lookup.value: {bc_lookup.value.get('India')}")

In [ ]:
# Use broadcast variable in a UDF (accesses it from executor memory)
@udf(StringType())
def get_continent(country):
    if country is None:
        return "Unknown"
    return bc_lookup.value.get(country, "Other")

customers = spark.read.csv("/workspace/week4/data/customers.csv",
                            header=True, inferSchema=True)

result = customers.withColumn("continent", get_continent(F.col("country")))
result.select("name", "country", "continent").show()

In [ ]:
# Performance: broadcast vs closure capture
# In local mode, both go to the same JVM — but in real clusters the difference is huge

big_lookup = {str(i): f"value_{i}" for i in range(100_000)}  # ~5MB dict

# Method 1: closure capture — Python dict captured by lambda, serialized per task
t0 = time.time()
@udf(StringType())
def lookup_closure(key):  # big_lookup captured in closure
    return big_lookup.get(key)
df_keys = spark.range(50_000).withColumn("key", (F.col("id") % 100_000).cast("string"))
df_keys.withColumn("val", lookup_closure(F.col("key"))).count()
closure_time = time.time() - t0

# Method 2: broadcast variable — dict sent once per executor
bc_big = sc.broadcast(big_lookup)
t0 = time.time()
@udf(StringType())
def lookup_broadcast(key):
    return bc_big.value.get(key)
df_keys.withColumn("val", lookup_broadcast(F.col("key"))).count()
broadcast_time = time.time() - t0

print(f"Closure capture:    {closure_time:.3f}s")
print(f"Broadcast variable: {broadcast_time:.3f}s")
print("(Difference more dramatic on real clusters with many tasks)")

# Always unpersist broadcast variables when done
bc_big.unpersist()
bc_lookup.unpersist()

## Part 2: Broadcast Variables for ML Model Weights

In [ ]:
# Classic pattern: broadcast a trained ML model to score data at scale
import numpy as np

# Simulate a trained model (weights array)
model_weights = np.random.randn(10).astype(np.float32)  # 10 feature weights
model_bias    = 0.5

# Broadcast the model
bc_model = sc.broadcast({"weights": model_weights, "bias": model_bias})

# Create feature data
import numpy as np
feature_df = spark.range(10_000) \
    .withColumn("f0", F.rand()).withColumn("f1", F.rand()) \
    .withColumn("f2", F.rand()).withColumn("f3", F.rand()) \
    .withColumn("f4", F.rand())

# Pandas UDF for efficient scoring with the broadcast model
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(DoubleType())
def score(f0: pd.Series, f1: pd.Series, f2: pd.Series,
          f3: pd.Series, f4: pd.Series) -> pd.Series:
    model = bc_model.value
    features = np.column_stack([f0, f1, f2, f3, f4])
    # Linear model: dot product of features with first 5 weights + bias
    scores = features @ model["weights"][:5] + model["bias"]
    return pd.Series(scores)

scored = feature_df.withColumn("score", score("f0", "f1", "f2", "f3", "f4"))
print("Model scoring with broadcast weights:")
scored.select("id", "score").show(5)

bc_model.unpersist()

## Part 3: Accumulators — Counting Across Executors

In [ ]:
# Built-in long accumulator
total_rows     = sc.accumulator(0)
null_rows      = sc.accumulator(0)
negative_vals  = sc.accumulator(0)

# Dataset with some bad data
raw_data = spark.createDataFrame([
    (1, "Alice",  95000),
    (2, "Bob",    88000),
    (3, None,     72000),    # null name
    (4, "Dave",   -5000),    # negative salary!
    (5, "Eve",    None),     # null salary
    (6, "Frank",  65000),
    (7, None,     -100),     # both bad!
    (8, "Grace",  80000),
], ["id", "name", "salary"])

def validate_and_count(row):
    total_rows.add(1)
    has_problem = False
    if row.name is None or row.salary is None:
        null_rows.add(1)
        has_problem = True
    if row.salary is not None and row.salary < 0:
        negative_vals.add(1)
        has_problem = True
    return not has_problem  # True = keep the row

# Apply validation via RDD (accumulators only reliable in actions)
cleaned_rdd = raw_data.rdd.filter(validate_and_count)
cleaned_df  = spark.createDataFrame(cleaned_rdd, raw_data.schema)

# Trigger action to materialize the accumulators
valid_count = cleaned_df.count()

print(f"Total rows processed:  {total_rows.value}")
print(f"Rows with nulls:       {null_rows.value}")
print(f"Rows with negative:    {negative_vals.value}")
print(f"Valid rows kept:       {valid_count}")

cleaned_df.show()

In [ ]:
# Custom accumulator — collect error messages (not just counts)
from pyspark import AccumulatorParam

class ListAccumulatorParam(AccumulatorParam):
    def zero(self, initial_value):
        return []
    def addInPlace(self, v1, v2):
        v1.extend(v2)
        return v1

error_messages = sc.accumulator([], ListAccumulatorParam())

def transform_with_errors(row):
    errors = []
    if row.name is None:
        errors.append(f"id={row.id}: NULL name")
    if row.salary is not None and row.salary < 0:
        errors.append(f"id={row.id}: negative salary {row.salary}")
    if errors:
        error_messages.add(errors)
        return None  # mark as invalid
    return row

valid = raw_data.rdd.map(transform_with_errors).filter(lambda r: r is not None)
valid.count()  # trigger

print("Collected error messages:")
for msg in error_messages.value:
    print(f"  {msg}")

print()
print("WARNING: List accumulators grow unboundedly. Use for debugging only,")
print("not in production with millions of error rows!")

In [ ]:
# Accumulator in DataFrame operations with foreachPartition
# foreachPartition is an action → accumulators reliable

processed_count = sc.accumulator(0)
error_count     = sc.accumulator(0)

orders = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)

def process_partition(partition_rows):
    for row in partition_rows:
        processed_count.add(1)
        if row.amount is None or row.amount <= 0:
            error_count.add(1)

orders.foreachPartition(process_partition)   # action → reliable accumulator update

print(f"Processed: {processed_count.value} rows")
print(f"Errors:    {error_count.value} rows")

## Part 4: Accumulator Gotchas

In [ ]:
print("""
Accumulator Gotchas:
═══════════════════════════════════════════════════════════════
Gotcha 1: Transformations ≠ reliable accumulator updates

  counter = sc.accumulator(0)

  # BAD — transformation (lazy, may run multiple times on retry)
  rdd = data.rdd.map(lambda r: (counter.add(1), r)[1])  # called per-retry!
  rdd.count()    # 1st run: counter = 100
  rdd.count()    # 2nd run: counter = 200 (counted twice!)

  # GOOD — action (each task counted exactly once)
  data.rdd.foreach(lambda r: counter.add(1))  # action, not transformation

Gotcha 2: Speculative execution double-counts
  If spark.speculation=true, slow tasks are re-launched.
  Both the original and speculative task update the accumulator.
  Result: over-counting by number of speculated tasks.

Gotcha 3: Task retries (on failure) add again
  If a task fails and retries, it re-adds to the accumulator.
  For exactly-once counting: use external systems (metrics server,
  Kafka, DB transactions) — accumulators are best-effort.

When accumulators ARE reliable:
  ✓ Debugging/monitoring (approximate counts are fine)
  ✓ Progress reporting during long jobs
  ✓ foreachPartition / foreach actions (not retried by default)
  ✗ Financial calculations, compliance counts (use DB transactions)
═══════════════════════════════════════════════════════════════
""")

## Exercises

1. Create a broadcast variable containing a dict of `product_id → category`. Use it in a UDF to enrich an orders DataFrame without a join.
2. Rewrite Exercise 1 using a broadcast DataFrame join instead. Why is the join approach generally better than a UDF with broadcast?
3. Create accumulators to count: total rows, rows with `status='cancelled'`, and rows with `amount > 1000`. Apply them to the orders dataset.
4. Why can't tasks READ an accumulator value? (Hint: think about what would happen if 100 tasks tried to read and act on the same counter simultaneously.)
5. What is the maximum safe size for a broadcast variable? What happens if you exceed it?

In [ ]:
# Exercise 1 & 2: Broadcast lookup vs join
products = spark.read.csv("/workspace/week4/data/products.csv", header=True, inferSchema=True)
orders   = spark.read.csv("/workspace/week4/data/orders.csv",   header=True, inferSchema=True)

# Build lookup dict
product_category = {row.product_id: row.category for row in products.collect()}
bc_cat = sc.broadcast(product_category)

# Method 1: UDF with broadcast
@udf(StringType())
def get_category(pid):
    return bc_cat.value.get(pid, "Unknown") if pid else "Unknown"

print("Method 1: UDF with broadcast variable")
orders.withColumn("category", get_category(F.col("product_id"))) \
      .select("order_id", "product_id", "category") \
      .show(5)

# Method 2: Broadcast DataFrame join (better — Catalyst optimizes, no UDF overhead)
print("Method 2: Broadcast DataFrame join (preferred)")
from pyspark.sql.functions import broadcast as bc_hint
orders.join(bc_hint(products.select("product_id", "category")),
            on="product_id", how="left") \
      .select("order_id", "product_id", "category") \
      .show(5)

bc_cat.unpersist()

In [ ]:
# Exercise 3: Three accumulators on orders
total     = sc.accumulator(0)
cancelled = sc.accumulator(0)
high_val  = sc.accumulator(0)

def count_metrics(row):
    total.add(1)
    if row.status == "cancelled":
        cancelled.add(1)
    if row.amount is not None and row.amount > 1000:
        high_val.add(1)

orders.rdd.foreach(count_metrics)

print(f"Total orders:          {total.value}")
print(f"Cancelled orders:      {cancelled.value}")
print(f"High-value (>$1000):   {high_val.value}")